In [1]:
# Step 0: Import Required Libraries & Resource Downloads
import sys
import re
import string
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import stopwordsiso

# Safely reconfigure stdout for UTF-8 encoding if supported by stream
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Download essential NLTK packages
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✅ All required libraries imported and NLTK resources downloaded successfully.")
print(f"English Stopwords available: {len(stopwordsiso.stopwords('en'))}")
print(f"Marathi Stopwords available: {len(stopwordsiso.stopwords('mr'))}")
print(f"Hindi Stopwords available: {len(stopwordsiso.stopwords('hi'))}")


✅ All required libraries imported and NLTK resources downloaded successfully.
English Stopwords available: 1298
Marathi Stopwords available: 99
Hindi Stopwords available: 225


In [2]:
# Define English, Marathi, and Hindi Input Corpora
corpora = {
    "English": "Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence! It enables computers to understand, interpret, and manipulate human language in 2026. NLTK provides 100+ tools for text processing.",
    "Marathi": "नैसर्गिक भाषा प्रक्रिया (NLP) हा कृत्रिम बुद्धिमत्ता (AI) चा एक अतिशय महत्त्वाचा भाग आहे! २०२६ मध्ये संगणक मानवी भाषा सहजपणे समजतात. मराठी भाषेत विविध १००० पेक्षा जास्त शब्दसंग्रह समाविष्ट आहेत.",
    "Hindi": "प्राकृतिक भाषा प्रसंस्करण (NLP) कृत्रिम बुद्धिमत्ता (AI) का एक मुख्य क्षेत्र है! यह संगणक को मानव भाषा को समझने, विश्लेषण करने और 2026 में संशोधित करने की अनुमति देता है। इसमें 500 से अधिक तकनीकें शामिल हैं।"
}

for lang, text in corpora.items():
    print(f"==================== {lang.upper()} ORIGINAL CORPUS ====================")
    print(text)
    print()


==================== ENGLISH ORIGINAL CORPUS ====================
Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence! It enables computers to understand, interpret, and manipulate human language in 2026. NLTK provides 100+ tools for text processing.

==================== MARATHI ORIGINAL CORPUS ====================
नैसर्गिक भाषा प्रक्रिया (NLP) हा कृत्रिम बुद्धिमत्ता (AI) चा एक अतिशय महत्त्वाचा भाग आहे! २०२६ मध्ये संगणक मानवी भाषा सहजपणे समजतात. मराठी भाषेत विविध १००० पेक्षा जास्त शब्दसंग्रह समाविष्ट आहेत.

==================== HINDI ORIGINAL CORPUS ====================
प्राकृतिक भाषा प्रसंस्करण (NLP) कृत्रिम बुद्धिमत्ता (AI) का एक मुख्य क्षेत्र है! यह संगणक को मानव भाषा को समझने, विश्लेषण करने और 2026 में संशोधित करने की अनुमति देता है। इसमें 500 से अधिक तकनीकें शामिल हैं।



In [3]:
# Define Preprocessing Utility Functions for Multilingual Text Processing

# 1. Punctuation removal (Handles ASCII punctuation + Devanagari Purna Viram । and Double Danda ॥)
def remove_punctuation(text):
    punct_pattern = f"[{re.escape(string.punctuation)}।॥]"
    return re.sub(punct_pattern, "", text)

# 2. Number removal (Handles ASCII digits 0-9 and Devanagari digits ०-९)
def remove_numbers(text):
    return re.sub(r'[\d०-९]+', '', text)

# 3. Marathi Suffix Stripper (Stemmer)
marathi_suffixes = ['ांना', 'ांचे', 'ांच्या', 'ाला', 'ातील', 'मुळे', 'साठी', 'कडे', 'वर', 'ात', 'ने', 'चा', 'ची', 'चे', 'च्या', 'ातून', 'तील', 'ही', 'त']
def marathi_stem(word):
    for suffix in sorted(marathi_suffixes, key=len, reverse=True):
        if word.endswith(suffix) and len(word) - len(suffix) >= 2:
            return word[:-len(suffix)]
    return word

# 4. Marathi Lemmatizer (Morphological dictionary fallback + Stemmer)
def marathi_lemmatize(word):
    stemmed = marathi_stem(word)
    lemma_dict = {
        "संगणक": "संगणक", "भाषा": "भाषा", "प्रक्रिया": "प्रक्रिया",
        "बुद्धिमत्ता": "बुद्धिमत्ता", "शब्दसंग्रह": "शब्दसंग्रह"
    }
    return lemma_dict.get(stemmed, stemmed)

# 5. Hindi Suffix Stripper (Stemmer)
hindi_suffixes = ['ों', 'ियां', 'ाएं', 'ाओं', 'ना', 'ने', 'नी', 'कर', 'ते', 'ता', 'ती', 'ा', 'ी', 'े', 'ों', 'ओं']
def hindi_stem(word):
    for suffix in sorted(hindi_suffixes, key=len, reverse=True):
        if word.endswith(suffix) and len(word) - len(suffix) >= 2:
            return word[:-len(suffix)]
    return word

# 6. Hindi Lemmatizer (Morphological dictionary fallback + Stemmer)
def hindi_lemmatize(word):
    stemmed = hindi_stem(word)
    lemma_dict = {
        "तकनीकें": "तकनीक", "अनुमति": "अनुमति", "भाषा": "भाषा",
        "प्रसंस्करण": "प्रसंस्करण", "क्षेत्र": "क्षेत्र"
    }
    return lemma_dict.get(stemmed, stemmed)

print("✅ Utility functions initialized successfully.")


✅ Utility functions initialized successfully.


In [4]:
# English Pipeline Step-by-Step Execution
corpus_en = corpora["English"]

# Step 1: Original Corpus
print("--- Step 1: Original Corpus ---")
print(corpus_en)

# Step 2: Lowercasing
step2_en = corpus_en.lower()
print("\n--- Step 2: Lowercase Text ---")
print(step2_en)

# Step 3: Punctuation Removal
step3_en = remove_punctuation(step2_en)
print("\n--- Step 3: Punctuation Removed Text ---")
print(step3_en)

# Step 4: Number Removal
step4_en = remove_numbers(step3_en)
print("\n--- Step 4: Number Removed Text ---")
print(step4_en)

# Step 5: Tokenization
step5_en = word_tokenize(step4_en)
print("\n--- Step 5: Tokenized Text ---")
print(step5_en)

# Step 6: Stopword Removal using NLTK and stopwordsiso
en_stopwords = set(stopwords.words('english')).union(stopwordsiso.stopwords('en'))
step6_en = [w for w in step5_en if w.lower() not in en_stopwords and w.strip()]
print("\n--- Step 6: Stopword Removed Text ---")
print(step6_en)

# Step 7: Stemming using NLTK PorterStemmer
porter = PorterStemmer()
step7_en = [porter.stem(w) for w in step6_en]
print("\n--- Step 7: Stemmed Output ---")
print(step7_en)

# Step 8: Lemmatization using NLTK WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
step8_en = [lemmatizer.lemmatize(w) for w in step6_en]
print("\n--- Step 8: Lemmatized Output ---")
print(step8_en)

# Step 9: Final Preprocessed Text
step9_en = " ".join(step8_en)
print("\n--- Step 9: Final Preprocessed Text ---")
print(step9_en)


--- Step 1: Original Corpus ---
Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence! It enables computers to understand, interpret, and manipulate human language in 2026. NLTK provides 100+ tools for text processing.

--- Step 2: Lowercase Text ---
natural language processing (nlp) is a fascinating field of artificial intelligence! it enables computers to understand, interpret, and manipulate human language in 2026. nltk provides 100+ tools for text processing.

--- Step 3: Punctuation Removed Text ---
natural language processing nlp is a fascinating field of artificial intelligence it enables computers to understand interpret and manipulate human language in 2026 nltk provides 100 tools for text processing

--- Step 4: Number Removed Text ---
natural language processing nlp is a fascinating field of artificial intelligence it enables computers to understand interpret and manipulate human language in  nltk provides  tools for text processing

--- Step 5:


--- Step 8: Lemmatized Output ---
['natural', 'language', 'processing', 'nlp', 'fascinating', 'field', 'artificial', 'intelligence', 'enables', 'computer', 'understand', 'interpret', 'manipulate', 'human', 'language', 'nltk', 'tool', 'processing']

--- Step 9: Final Preprocessed Text ---
natural language processing nlp fascinating field artificial intelligence enables computer understand interpret manipulate human language nltk tool processing


In [5]:
# Marathi Pipeline Step-by-Step Execution
corpus_mr = corpora["Marathi"]

# Step 1: Original Corpus
print("--- Step 1: Original Corpus ---")
print(corpus_mr)

# Step 2: Lowercasing
step2_mr = corpus_mr.lower()
print("\n--- Step 2: Lowercase Text ---")
print(step2_mr)

# Step 3: Punctuation Removal (Includes Devanagari Purna Viram ।)
step3_mr = remove_punctuation(step2_mr)
print("\n--- Step 3: Punctuation Removed Text ---")
print(step3_mr)

# Step 4: Number Removal (Includes ASCII 0-9 and Devanagari digits ०-९)
step4_mr = remove_numbers(step3_mr)
print("\n--- Step 4: Number Removed Text ---")
print(step4_mr)

# Step 5: Tokenization
step5_mr = word_tokenize(step4_mr)
print("\n--- Step 5: Tokenized Text ---")
print(step5_mr)

# Step 6: Stopword Removal using stopwordsiso ('mr')
mr_stopwords = stopwordsiso.stopwords('mr')
step6_mr = [w for w in step5_mr if w.lower() not in mr_stopwords and w.strip()]
print("\n--- Step 6: Stopword Removed Text ---")
print(step6_mr)

# Step 7: Stemming using Marathi Suffix Stripping
step7_mr = [marathi_stem(w) for w in step6_mr]
print("\n--- Step 7: Stemmed Output ---")
print(step7_mr)

# Step 8: Lemmatization using Marathi Morphological Rules
step8_mr = [marathi_lemmatize(w) for w in step6_mr]
print("\n--- Step 8: Lemmatized Output ---")
print(step8_mr)

# Step 9: Final Preprocessed Text
step9_mr = " ".join(step8_mr)
print("\n--- Step 9: Final Preprocessed Text ---")
print(step9_mr)


--- Step 1: Original Corpus ---
नैसर्गिक भाषा प्रक्रिया (NLP) हा कृत्रिम बुद्धिमत्ता (AI) चा एक अतिशय महत्त्वाचा भाग आहे! २०२६ मध्ये संगणक मानवी भाषा सहजपणे समजतात. मराठी भाषेत विविध १००० पेक्षा जास्त शब्दसंग्रह समाविष्ट आहेत.

--- Step 2: Lowercase Text ---
नैसर्गिक भाषा प्रक्रिया (nlp) हा कृत्रिम बुद्धिमत्ता (ai) चा एक अतिशय महत्त्वाचा भाग आहे! २०२६ मध्ये संगणक मानवी भाषा सहजपणे समजतात. मराठी भाषेत विविध १००० पेक्षा जास्त शब्दसंग्रह समाविष्ट आहेत.

--- Step 3: Punctuation Removed Text ---
नैसर्गिक भाषा प्रक्रिया nlp हा कृत्रिम बुद्धिमत्ता ai चा एक अतिशय महत्त्वाचा भाग आहे २०२६ मध्ये संगणक मानवी भाषा सहजपणे समजतात मराठी भाषेत विविध १००० पेक्षा जास्त शब्दसंग्रह समाविष्ट आहेत

--- Step 4: Number Removed Text ---
नैसर्गिक भाषा प्रक्रिया nlp हा कृत्रिम बुद्धिमत्ता ai चा एक अतिशय महत्त्वाचा भाग आहे  मध्ये संगणक मानवी भाषा सहजपणे समजतात मराठी भाषेत विविध  पेक्षा जास्त शब्दसंग्रह समाविष्ट आहेत

--- Step 5: Tokenized Text ---
['नैसर्गिक', 'भाषा', 'प्रक्रिया', 'nlp', 'हा', 'कृत्रिम', 'बुद्धिमत

In [6]:
# Hindi Pipeline Step-by-Step Execution
corpus_hi = corpora["Hindi"]

# Step 1: Original Corpus
print("--- Step 1: Original Corpus ---")
print(corpus_hi)

# Step 2: Lowercasing
step2_hi = corpus_hi.lower()
print("\n--- Step 2: Lowercase Text ---")
print(step2_hi)

# Step 3: Punctuation Removal (Includes Devanagari Purna Viram ।)
step3_hi = remove_punctuation(step2_hi)
print("\n--- Step 3: Punctuation Removed Text ---")
print(step3_hi)

# Step 4: Number Removal (Includes ASCII 0-9 and Devanagari digits ०-९)
step4_hi = remove_numbers(step3_hi)
print("\n--- Step 4: Number Removed Text ---")
print(step4_hi)

# Step 5: Tokenization
step5_hi = word_tokenize(step4_hi)
print("\n--- Step 5: Tokenized Text ---")
print(step5_hi)

# Step 6: Stopword Removal using stopwordsiso ('hi')
hi_stopwords = stopwordsiso.stopwords('hi')
step6_hi = [w for w in step5_hi if w.lower() not in hi_stopwords and w.strip()]
print("\n--- Step 6: Stopword Removed Text ---")
print(step6_hi)

# Step 7: Stemming using Hindi Suffix Stripping
step7_hi = [hindi_stem(w) for w in step6_hi]
print("\n--- Step 7: Stemmed Output ---")
print(step7_hi)

# Step 8: Lemmatization using Hindi Morphological Rules
step8_hi = [hindi_lemmatize(w) for w in step6_hi]
print("\n--- Step 8: Lemmatized Output ---")
print(step8_hi)

# Step 9: Final Preprocessed Text
step9_hi = " ".join(step8_hi)
print("\n--- Step 9: Final Preprocessed Text ---")
print(step9_hi)


--- Step 1: Original Corpus ---
प्राकृतिक भाषा प्रसंस्करण (NLP) कृत्रिम बुद्धिमत्ता (AI) का एक मुख्य क्षेत्र है! यह संगणक को मानव भाषा को समझने, विश्लेषण करने और 2026 में संशोधित करने की अनुमति देता है। इसमें 500 से अधिक तकनीकें शामिल हैं।

--- Step 2: Lowercase Text ---
प्राकृतिक भाषा प्रसंस्करण (nlp) कृत्रिम बुद्धिमत्ता (ai) का एक मुख्य क्षेत्र है! यह संगणक को मानव भाषा को समझने, विश्लेषण करने और 2026 में संशोधित करने की अनुमति देता है। इसमें 500 से अधिक तकनीकें शामिल हैं।

--- Step 3: Punctuation Removed Text ---
प्राकृतिक भाषा प्रसंस्करण nlp कृत्रिम बुद्धिमत्ता ai का एक मुख्य क्षेत्र है यह संगणक को मानव भाषा को समझने विश्लेषण करने और 2026 में संशोधित करने की अनुमति देता है इसमें 500 से अधिक तकनीकें शामिल हैं

--- Step 4: Number Removed Text ---
प्राकृतिक भाषा प्रसंस्करण nlp कृत्रिम बुद्धिमत्ता ai का एक मुख्य क्षेत्र है यह संगणक को मानव भाषा को समझने विश्लेषण करने और  में संशोधित करने की अनुमति देता है इसमें  से अधिक तकनीकें शामिल हैं

--- Step 5: Tokenized Text ---
['प्राकृतिक', 'भ

In [7]:
# Construct Summary DataFrame
summary_data = {
    "Language": ["English 🇬🇧", "Marathi 🇮🇳", "Hindi 🇮🇳"],
    "Original Corpus": [corpora["English"], corpora["Marathi"], corpora["Hindi"]],
    "Final Preprocessed Text": [step9_en, step9_mr, step9_hi]
}

df_summary = pd.DataFrame(summary_data)
pd.set_option('display.max_colwidth', None)
df_summary


,Language,Original Corpus,Final Preprocessed Text
0,English 🇬🇧,"Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence! It enables computers to understand, interpret, and manipulate human language in 2026. NLTK provides 100+ tools for text processing.",natural language processing nlp fascinating field artificial intelligence enables computer understand interpret manipulate human language nltk tool processing
1,Marathi 🇮🇳,नैसर्गिक भाषा प्रक्रिया (NLP) हा कृत्रिम बुद्धिमत्ता (AI) चा एक अतिशय महत्त्वाचा भाग आहे! २०२६ मध्ये संगणक मानवी भाषा सहजपणे समजतात. मराठी भाषेत विविध १००० पेक्षा जास्त शब्दसंग्रह समाविष्ट आहेत.,नैसर्गिक भाषा प्रक्रिया nlp कृत्रिम बुद्धिमत्ता ai चा अतिशय महत्त्वा भाग मध्ये संगणक मानवी भाषा सहजपणे समजत मराठी भाषे विविध पेक्षा जास् शब्दसंग्रह समाविष्ट
2,Hindi 🇮🇳,"प्राकृतिक भाषा प्रसंस्करण (NLP) कृत्रिम बुद्धिमत्ता (AI) का एक मुख्य क्षेत्र है! यह संगणक को मानव भाषा को समझने, विश्लेषण करने और 2026 में संशोधित करने की अनुमति देता है। इसमें 500 से अधिक तकनीकें शामिल हैं।",प्राकृतिक भाष प्रसंस्करण nlp कृत्रिम बुद्धिमत् ai मुख्य क्षेत्र संगणक मानव भाष समझ विश्लेषण संशोधित अनुमति दे अधिक तकनीक शामिल
